# Teste de Inferência ONNX - FingerNet

Este notebook compara os modelos ONNX do FingerNet:
1. **Modelo sem minúcias**: Apenas segmentação + enhancement
2. **Modelo completo**: Com extração de minúcias

Vamos cronometrar o tempo de inferência em CPU e visualizar os resultados.

In [ ]:
import sys
import os
import time
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

# Adiciona o diretório pai ao path para importar fingernet
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

import onnxruntime as ort
from fingernet.wrapper import postprocess
from fingernet.plot import plot_mnt, plot_ori_field, plot_img

# Configurar ONNX Runtime para usar APENAS 1 CORE/THREAD
sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = 1  # 1 thread para operações dentro de cada nó
sess_options.inter_op_num_threads = 1  # 1 thread para paralelização entre nós
sess_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL  # Execução sequencial

print("⚙️  Configuração: Single-core/thread")
print(f"   - intra_op_num_threads: {sess_options.intra_op_num_threads}")
print(f"   - inter_op_num_threads: {sess_options.inter_op_num_threads}")
print(f"   - execution_mode: SEQUENTIAL")

## 1. Carregar Imagem de Teste

In [ ]:
# Caminho da imagem
image_path = "/localStorage/data/datasets/SD258/orig/000/00/sd258_000_11-00_latent_bad.png"

# Carregar e preprocessar imagem
img_pil = Image.open(image_path).convert("L")
img_np = np.array(img_pil, dtype=np.float32) / 255.0
original_img = img_np.copy()

# Adicionar padding para múltiplo de 8
h, w = img_np.shape
pad_h = (8 - h % 8) % 8
pad_w = (8 - w % 8) % 8
img_padded = np.pad(img_np, ((0, pad_h), (0, pad_w)), mode='constant', constant_values=0)

# Converter para formato ONNX (NCHW)
input_tensor = img_padded[np.newaxis, np.newaxis, :, :].astype(np.float32)

print(f"Imagem original: {img_np.shape}")
print(f"Imagem padded: {img_padded.shape}")
print(f"Input tensor: {input_tensor.shape}")

# Visualizar imagem
plt.figure(figsize=(8, 6))
plt.imshow(original_img, cmap='gray')
plt.title("Imagem Original")
plt.axis('off')
plt.show()

## 2. Inferência - Modelo SEM Minúcias (Seg + Enhancement)

In [ ]:
# Carregar modelo ONNX sem minúcias (usando sess_options para single-core)
onnx_seg_path = "fingernet_seg_enh.onnx"
session_seg = ort.InferenceSession(onnx_seg_path, sess_options=sess_options, providers=['CPUExecutionProvider'])

# Warmup (para garantir tempo preciso)
for _ in range(3):
    _ = session_seg.run(None, {'input_image': input_tensor})

# Cronometrar inferência (10 iterações para precisão)
times_seg = []
for _ in range(10):
    start = time.perf_counter()
    outputs_seg = session_seg.run(None, {'input_image': input_tensor})
    end = time.perf_counter()
    times_seg.append(end - start)

avg_time_seg = np.mean(times_seg) * 1000  # Converter para ms
std_time_seg = np.std(times_seg) * 1000

print(f"Modelo SEM Minúcias (SINGLE-CORE):")
print(f"  Tempo médio: {avg_time_seg:.2f} ± {std_time_seg:.2f} ms")
print(f"  FPS: {1000/avg_time_seg:.2f}")
print(f"  Outputs: {len(outputs_seg)}")
print(f"    - segmentation: {outputs_seg[0].shape}")
print(f"    - enhanced_image: {outputs_seg[1].shape}")
print(f"    - orientation: {outputs_seg[2].shape}")

In [ ]:
# Visualizar outputs do modelo sem minúcias
seg_map = outputs_seg[0][0, 0, :h, :w]  # Remove padding
enh_img = outputs_seg[1][0, 0, :h, :w]
ori_map = outputs_seg[2][0, :, :h, :w]

# Processar orientação
ori_idx = np.argmax(ori_map, axis=0)
orientation_field = (ori_idx * 2.0 - 89.) * np.pi / 180.0

# Processar máscara (binarizar)
seg_binary = (np.round(seg_map) > 0.5).astype(np.float32)

# Processar imagem melhorada (normalizar para visualização)
enh_vis = enh_img * seg_binary
enh_min, enh_max = enh_vis.min(), enh_vis.max()
enh_norm = ((enh_vis - enh_min) / (enh_max - enh_min + 1e-8) * 255).astype(np.uint8)

# Plotar
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(seg_binary, cmap='gray')
axes[0].set_title("Segmentação")
axes[0].axis('off')

axes[1].imshow(enh_norm, cmap='gray')
axes[1].set_title("Imagem Melhorada")
axes[1].axis('off')

axes[2].imshow(original_img, cmap='gray')
plot_ori_field(axes[2], orientation_field, stride=16)
axes[2].set_title("Campo de Orientação")
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 3. Inferência - Modelo COMPLETO (Com Minúcias)

In [ ]:
# Carregar modelo ONNX completo (usando sess_options para single-core)
onnx_full_path = "fingernet.onnx"
session_full = ort.InferenceSession(onnx_full_path, sess_options=sess_options, providers=['CPUExecutionProvider'])

# Warmup
for _ in range(3):
    _ = session_full.run(None, {'input_image': input_tensor})

# Cronometrar inferência
times_full = []
for _ in range(10):
    start = time.perf_counter()
    outputs_full = session_full.run(None, {'input_image': input_tensor})
    end = time.perf_counter()
    times_full.append(end - start)

avg_time_full = np.mean(times_full) * 1000
std_time_full = np.std(times_full) * 1000

print(f"Modelo COMPLETO (com minúcias - SINGLE-CORE):")
print(f"  Tempo médio: {avg_time_full:.2f} ± {std_time_full:.2f} ms")
print(f"  FPS: {1000/avg_time_full:.2f}")
print(f"  Outputs: {len(outputs_full)}")
print(f"    - orientation_upsample: {outputs_full[0].shape}")
print(f"    - segmentation_upsample: {outputs_full[1].shape}")
print(f"    - enhanced_real: {outputs_full[2].shape}")
print(f"    - minutiae_orientation: {outputs_full[3].shape}")
print(f"    - minutiae_x_offset: {outputs_full[4].shape}")
print(f"    - minutiae_y_offset: {outputs_full[5].shape}")
print(f"    - minutiae_score: {outputs_full[6].shape}")

print(f"\n⏱️  COMPARAÇÃO:")
print(f"  Speedup sem minúcias: {avg_time_full/avg_time_seg:.2f}x mais rápido")
print(f"  Overhead de minúcias: {avg_time_full - avg_time_seg:.2f} ms")

## 4. Conversão de Minúcias (Post-processing)

In [ ]:
# Converter outputs ONNX para formato PyTorch para usar a função postprocess
outputs_dict = {
    'orientation upsample': torch.from_numpy(outputs_full[0]),
    'segmentation upsample': torch.from_numpy(outputs_full[1]),
    'segmentation': torch.from_numpy(outputs_full[1][:, :, ::8, ::8]),  # Downsample
    'orientation': torch.from_numpy(outputs_full[0][:, :, ::8, ::8]),
    'enhanced_real': torch.from_numpy(outputs_full[2]),
    'minutiae_orientation': torch.from_numpy(outputs_full[3]),
    'minutiae_x_offset': torch.from_numpy(outputs_full[4]),
    'minutiae_y_offset': torch.from_numpy(outputs_full[5]),
    'minutiae_score': torch.from_numpy(outputs_full[6])
}

# Aplicar postprocessing (mesma função usada no wrapper)
threshold = 0.5
processed = postprocess(outputs_dict, threshold)

# Extrair minúcias (lista de tensors, um por imagem no batch)
minutiae_list = processed['minutiae']
minutiae_np = minutiae_list[0].cpu().numpy()  # Primeira (e única) imagem

# Remover padding das outras saídas
enhanced_image = processed['enhanced_image'][0, :h, :w].cpu().numpy()
segmentation_mask = processed['segmentation_mask'][0, :h, :w].cpu().numpy()
orientation_field_full = processed['orientation_field'][0, :h, :w].cpu().numpy()

print(f"Minúcias detectadas: {len(minutiae_np)}")
print(f"Enhanced image shape: {enhanced_image.shape}")
print(f"Segmentation mask shape: {segmentation_mask.shape}")
print(f"Orientation field shape: {orientation_field_full.shape}")

# Mostrar primeiras 5 minúcias
if len(minutiae_np) > 0:
    print(f"\nPrimeiras 5 minúcias (x, y, angle, score):")
    for i, mnt in enumerate(minutiae_np[:5]):
        print(f"  {i+1}. x={mnt[0]:.1f}, y={mnt[1]:.1f}, angle={mnt[2]:.3f} rad, score={mnt[3]:.3f}")

## 5. Visualização Final - Modelo Completo

In [ ]:
# Plotar todos os resultados do modelo completo
fig, axes = plt.subplots(2, 2, figsize=(16, 16))

# Subplot 1: Imagem original
axes[0, 0].imshow(original_img, cmap='gray')
axes[0, 0].set_title("Imagem Original", fontsize=14)
axes[0, 0].axis('off')

# Subplot 2: Segmentação
axes[0, 1].imshow(segmentation_mask, cmap='gray')
axes[0, 1].set_title("Segmentação", fontsize=14)
axes[0, 1].axis('off')

# Subplot 3: Imagem melhorada
axes[1, 0].imshow(enhanced_image, cmap='gray')
axes[1, 0].set_title("Imagem Melhorada", fontsize=14)
axes[1, 0].axis('off')

# Subplot 4: Minúcias detectadas
axes[1, 1].imshow(original_img, cmap='gray')
if len(minutiae_np) > 0:
    plot_mnt(axes[1, 1], minutiae_np, r=15)
axes[1, 1].set_title(f"Minúcias Detectadas ({len(minutiae_np)})", fontsize=14)
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Campo de orientação sobreposto
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.imshow(original_img, cmap='gray')
plot_ori_field(ax, orientation_field_full, stride=16)
ax.set_title("Campo de Orientação", fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Resumo dos Resultados

In [ ]:
print("="*60)
print("RESUMO - INFERÊNCIA ONNX EM SINGLE-CORE CPU")
print("="*60)
print(f"\nImagem: {os.path.basename(image_path)}")
print(f"Tamanho: {original_img.shape}")
print()
print(f"1️⃣  Modelo SEM Minúcias (Seg + Enhancement):")
print(f"   ⏱️  Tempo: {avg_time_seg:.2f} ± {std_time_seg:.2f} ms")
print(f"   🚀 FPS: {1000/avg_time_seg:.2f}")
print()
print(f"2️⃣  Modelo COMPLETO (Com Minúcias):")
print(f"   ⏱️  Tempo: {avg_time_full:.2f} ± {std_time_full:.2f} ms")
print(f"   🚀 FPS: {1000/avg_time_full:.2f}")
print(f"   📍 Minúcias: {len(minutiae_np)}")
print()
print(f"📊 COMPARAÇÃO:")
print(f"   Modelo sem minúcias é {avg_time_full/avg_time_seg:.2f}x mais rápido")
print(f"   Overhead da extração de minúcias: {avg_time_full - avg_time_seg:.2f} ms")
print(f"   ({(avg_time_full - avg_time_seg)/avg_time_full*100:.1f}% do tempo total)")
print("="*60)